# Contrastive rewrite — qwen / sarcasm

Two DPO arms on identical prompts and identical rejected responses, differing only
in where the *chosen* response comes from:

| arm | chosen |
|---|---|
| baseline | GLM-4.5-Air writing from scratch with the constitution in context |
| `_hybrid` | free generation on the 500 constitution prompts, qwen minimally editing its own response on the 1330 LIMA prompts |

Cells are order-dependent: 3 creates the qwen column, 5 needs it, 7 needs 5.

In [ ]:
import os

# HF_HOME must be set before anything imports huggingface_hub
os.environ["HF_TOKEN"]    = "hf_..."
os.environ["WANDB_TOKEN"] = "..."          # 40 chars, not the setup placeholder
os.environ["HF_USER"]     = "invi-bhagyesh"
os.environ["HF_HOME"]     = "/workspace/.cache/huggingface"
os.chdir("/workspace/OpenCharacterTraining")

C, M, GPU = "sarcasm", "qwen-2.5-7b-it", "0"

In [ ]:
!git fetch origin && git checkout contrastive-rewrite && git log --oneline -1
!python run_all.py --model qwen --download-models

## 1. Baseline arm

Seeds the teacher responses from HF, generates qwen's rejected responses, formats to `data/dpo/`.

In [ ]:
!CUDA_VISIBLE_DEVICES=$GPU python run_data.py --stage dpo --model $M --constitution $C

## 2. Is the length gap real?

If chosen is systematically longer, DPO learns "longer = better" independent of content.
This number is how much the rewrite is worth.

In [ ]:
import pandas as pd

d = pd.read_json(f"data/dpo/{M}/{C}.jsonl", lines=True)
c = d["chosen"].apply(lambda m: len(m[1]["content"]))
r = d["rejected"].apply(lambda m: len(m[1]["content"]))
print(f"{len(d)} pairs | chosen {c.mean():.0f} chars, rejected {r.mean():.0f}, "
      f"chosen longer in {(c > r).mean():.0%}")

## 3. Rewrite pass

qwen edits its own responses — the paper's conditional rewrite uses the same model as
the original, so there is no teacher style to leak.

In [ ]:
!CUDA_VISIBLE_DEVICES=$GPU python -m character.distillation.teacher \
    --model $M --mode rewrite --student $M --constitution $C

## 4. Did it actually edit minimally?

Mostly `unchanged` means the trait isn't landing. Mostly dropped on length means the
model regenerated instead of editing — loosen `--max-ratio` or use a bigger rewriter.

In [ ]:
d = pd.read_json(f"data/distillation/{C}.jsonl", lines=True)
col = f"rewrite_{M}"
print(f"{d[col].notna().sum()} / {d[M].notna().sum()} rewrites kept\n")

row = d[d[col].notna()].iloc[0]
for k in ["prompt", M, col]:
    print(f"--- {k} ---\n{row[k][:400]}\n")

## 5. Hybrid arm

In [ ]:
!CUDA_VISIBLE_DEVICES=$GPU python run_data.py --stage dpo --model $M --constitution $C \
    --chosen-source hybrid

## 6. Train both arms

Each writes to its own adapters, checkpoints and HF repo, so they never collide.
Watch `df -h` — DeepSpeed checkpoints are ~30 GB each.

In [ ]:
# one GPU, so the arms run back to back. detached, to survive a kernel restart.
# on a multi-GPU pod, give each arm its own CUDA_VISIBLE_DEVICES and OCT_MASTER_PORT
# and run them concurrently instead.
open("run_arms.sh", "w").write(f"""#!/bin/bash
export CUDA_VISIBLE_DEVICES={GPU}
python run_all.py --model qwen --constitution {C} --stage dpo             > log_base.txt   2>&1
python run_all.py --model qwen --constitution {C} --stage dpo --arm _hybrid > log_hybrid.txt 2>&1
echo ALL DONE
""")

!nohup bash run_arms.sh > run_arms.log 2>&1 &

In [ ]:
!tail -3 log_base.txt log_hybrid.txt
!df -h /workspace
!du -sh /workspace/ckpt/* 2>/dev/null